## Initial Setup

### Libraries and Dependencies

In [52]:
!pip -q install \
  langgraph==0.2.39 \
  langchain==0.3.14 \
  langchain-core==0.3.29 \
  langchain-community==0.3.14 \
  langsmith==0.1.147 \
  openai==1.59.6 \
  langchain-openai==0.2.14 \
  pydantic==2.10.4 \
  pandas==2.2.3 \
  numpy==2.2.1

### Imports

In [53]:
import os              # environment variables / runtime config
import re              # text normalization + lightweight pattern matching
import json            # serialize/parse tool outputs + structured data interchange
import sqlite3         # connect/query the SQLite database
import pandas as pd    # load/query CSV reference data
import numpy as np     # numeric utilities
from typing import Any, Dict, List, Optional, TypedDict, Tuple  # type hints + structured state for agent workflow

from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage  # chat + tool observation messages
from langchain_core.prompts import ChatPromptTemplate  # consistent prompt templates
from langchain_core.tools import tool                 # define LLM-callable tools
from langchain_core.output_parsers import JsonOutputParser  # parse LLM outputs into JSON

from langchain_openai import ChatOpenAI  # OpenAI chat model wrapper
from langgraph.graph import StateGraph, END  # build the LangGraph state machine + termination node
from IPython.display import Image, display  # visualize the graph / notebook-friendly display

### Generic non-tool utility functions

In [54]:
def readCSV(filePath: str) -> pd.DataFrame:
    """Reads a CSV file and returns its content as a data frame.
        Inputs:
            filePath (str): The file path to the CSV file (e.g., "data/patients.csv").
        Output:
            pandas.DataFrame: A DataFrame containing the data from the CSV file.     
    """
    print(f"Attempting to read CSV file: {filePath}")
    frame = pd.read_csv(filePath) # Read the CSV file into a DataFrame
    # print(frame.head(1))  # Display the first few rows for verification
    print(f"Succesfully read {len(frame)} rows from {filePath}") # Log the number of rows read
    return frame

def readDB(db_path: str) -> sqlite3.Cursor:
    """Connects to the SQLite database and returns a cursor for executing queries.
        Inputs:
            db_path (str): The file path to the SQLite database (e.g., "ehr_database.db").
        Output: 
            sqlite3.Cursor: A cursor object that can be used to execute SQL queries on the database.
    """
    print(f"Connecting to database at: {db_path}")
    conn = sqlite3.connect(db_path)  # Establish a connection to the database
    cursor = conn.cursor()  # Create a cursor object to execute SQL queries
    return cursor

def queryDB(cursor: sqlite3.Cursor, query: str) -> List[Tuple]:
    """Executes a SQL query on the database and returns the results.
        Inputs:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            query (str): The SQL query to be executed (e.g., "SELECT * FROM patients;").
        Output:
            list: A list of tuples containing the results of the query.
    """
    print(f"Executing SQL query: {query}")
    cursor.execute(query)  # Execute the provided SQL query
    results = cursor.fetchall()  # Fetch all results from the executed query
    print(f"Query returned {len(results)} rows")  # Log the number of rows returned
    return results

def queryDBSafe(cursor: sqlite3.Cursor, query: str, params: tuple = ()) -> List[Tuple]:
    """Executes a Parameterized SQL query on the database and returns the results.
        This function is designed to safely execute SQL queries with parameters, preventing SQL injection attacks.
        Inputs:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            query (str): The SQL query to be executed, with placeholders for parameters (e.g., "SELECT * FROM patients WHERE age > ?").
            params (tuple): A tuple of parameters to be safely substituted into the query (e.g., (30,)).
        Output:
            list: A list of tuples containing the results of the query.
    """
    cursor.execute(query, params)  # Execute the provided SQL query
    results = cursor.fetchall()  # Fetch all results from the executed query
    print(f"Query returned {len(results)} rows")  # Log the number of rows returned
    return results

def tableCount(tableName: str, cursor: sqlite3.Cursor) -> int:
    """Counts the number of records in a specified table.
        Inputs:
        tableName (str): The name of the database table to count records from.
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        Output: 
            int: The count of records in the specified table.
    """
    query = f"SELECT COUNT(*) FROM {tableName};"  # SQL query to count records in the table
    print(f"Counting records in table: {tableName}")
    cursor.execute(query)  # Execute the count query
    count = cursor.fetchone()[0]  # Fetch the count result
    print(f"Table {tableName} has {count} records")  # Log the count result
    return count

def normalizeText(text: str) -> str:
    """Normalize text for consistent matching (lowercase + trimmed + whitespace-collapsed).
        Inputs:
        s (str): Any input text (will be cast to string).

    Output:
        str: Normalized text with:
            - leading/trailing whitespace removed
            - converted to lowercase
            - internal / multiple whitespace collapsed to single spaces
    """
    return re.sub(r"\s+", " ", str(text).strip().lower())  # Convert to lowercase and remove special characters

def toJson(obj: Any) -> str:
    """Convert an object to a JSON string.
        Inputs:
        obj (Any): Any Python object (dict/list/str/etc.). If the object contains
            non-JSON-serializable values (e.g., datetime, numpy types), they are
            converted to strings via default=str.

    Output:
        str: JSON-formatted string (UTF-8 friendly, ensure_ascii=False).
    """
    return json.dumps(obj, ensure_ascii=False, default=str)

### Define and load the provided CSV Data

In [55]:
TRUSTED_SOURCES_CSV = "datafiles/trusted_sources_catalog.csv"    # path to the trusted_sources_catalog.csv file
LAB_EXPLAIN_CSV =  "datafiles/patient_friendly_lab_explanations.csv"    # path to the patient_friendly_lab_explanations.csv file
MED_EDU_CSV =  "datafiles/medication_education.csv"    # path to the medication_education.csv file
POLICY_CSV =  "datafiles/safety_policy_rules.csv"    # path to the safety_policy_rules.csv file

trustedSources = readCSV(TRUSTED_SOURCES_CSV)
labExplain     = readCSV(LAB_EXPLAIN_CSV)
medEdu         = readCSV(MED_EDU_CSV)
policyRules    = readCSV(POLICY_CSV)

Attempting to read CSV file: datafiles/trusted_sources_catalog.csv
Succesfully read 20 rows from datafiles/trusted_sources_catalog.csv
Attempting to read CSV file: datafiles/patient_friendly_lab_explanations.csv
Succesfully read 30 rows from datafiles/patient_friendly_lab_explanations.csv
Attempting to read CSV file: datafiles/medication_education.csv
Succesfully read 30 rows from datafiles/medication_education.csv
Attempting to read CSV file: datafiles/safety_policy_rules.csv
Succesfully read 10 rows from datafiles/safety_policy_rules.csv


### Define and load the Database

In [56]:
DB_PATH = "datafiles/health_portal.db"    # path to the SQLite database file
cursor = readDB(DB_PATH)
allTables = queryDB(cursor, "SELECT name FROM sqlite_master WHERE type='table';")
for t in allTables:
    tableCount(t[0], cursor)
# Load the credentials JSON file and extract values

Connecting to database at: datafiles/health_portal.db
Executing SQL query: SELECT name FROM sqlite_master WHERE type='table';
Query returned 6 rows
Counting records in table: patients
Table patients has 10 records
Counting records in table: encounters
Table encounters has 29 records
Counting records in table: clinical_notes
Table clinical_notes has 43 records
Counting records in table: labs
Table labs has 141 records
Counting records in table: medications
Table medications has 56 records
Counting records in table: allergies
Table allergies has 15 records


### Define and load LLM Configuration and load it

In [57]:
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                               # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

modelGenerator = ChatOpenAI(model="gpt-40-mini", temperature=0.2)  # Initialize the OpenAI chat model with specified parameters
modelValidator = ChatOpenAI(model="gpt-40", temperature=0.0)  # Initialize a second OpenAI chat model for validation with zero temperature

## Agent and Tools

### Tools for the MCP Servers

#### `getPatientInfo`

In [58]:
@tool("getPatientInfo")
def getPatientInfo(cursor: sqlite3.Cursor, patient_id: str):
    """"
        Retrieves information about a patient from the database based on their unique identifier.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose information is being requested.

        Output:
            dict: A dictionary containing the patient's information, including:
                - patient_id
                - first_name
                - last_name
                - birth_year
"""

    rows = queryDBSafe(cursor, 
            """
            SELECT
            patient_id, first_name, last_name, birth_year, sex_at_birth,
            preferred_language, health_literacy_level, timezone, created_at
            FROM patients 
            WHERE patient_id = ?
            """, (patient_id))  # Execute a parameterized query to get patient info
    print(f"getPatientInfo: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows[0] if rows else {"error": f"Patient {patient_id} not found"})

#### `listPatientEncounters`

In [59]:
@tool("listPatientEncounters")
def listPatientEncounters(cursor: sqlite3.Cursor, patient_id: str, limit: int=5):
    """"
        Retrieves a list of encounters for a specific patient from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose encounters are being requested.
        
        Output:
            list: A list of dictionaries, each containing details about an encounter, including:
            - encounter_id
            - encounter_date
            - encounter_type
            - reason_for_visit
            - diagnosis_summary
            - provider_specialty
            - followup_instructions
            - care_team_contact
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
               encounter_id, encounter_date, encounter_type, reason_for_visit,
               diagnosis_summary, provider_specialty, followup_instructions, care_team_contact
            FROM encounters 
            WHERE patient_id = ?
            ORDER BY encounter_date DESC
            LIMIT ?
            """, (patient_id, max(1, min(int(limit), 10))))  # Execute a parameterized query to get patient encounters
    print(f"listPatientEncounters: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows)

#### `getRecentClinicalNote`

In [60]:
@tool("getRecentClinicalNotes")
def getRecentClinicalNotes(cursor: sqlite3.Cursor, patient_id: str, note_type: str = "visit_note"):
    """"
        Retrieves a list of recent clinical notes for a specific patient from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose clinical notes are being requested.
        
        Output:
            list: A list of dictionaries, each containing details about a clinical note, including:
            - note_id
            - encounter_id
            - note_date
            - author
            - note_content
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
               note_id, encounter_id, patient_id, note_type, note_text, created_at, author_role
            FROM clinical_notes 
            WHERE patient_id = ? AND note_type = ?
            ORDER BY created_at DESC
            LIMIT 1
            """, (patient_id, note_type))  # Execute a parameterized query to get recent clinical notes
    print(f"getRecentClinicalNotes: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows[0] if rows else {"error": f"No {note_type} found for patient {patient_id}"})

#### `getClinicalNotesForEncounter`

In [61]:
def getClinicalNotesForEncounter(cursor: sqlite3.Cursor, encounter_id: str):
    """"
        Retrieves all clinical notes associated with a specific encounter from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            encounter_id (str): The unique identifier of the encounter whose clinical notes are being requested.
        Output:
            list: A list of dictionaries, each containing details about a clinical note, including:
            - note_id
            - patient_id
            - note_type
            - note_content
            - created_at
            - author_role
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
                note_id, encounter_id, patient_id, note_type, note_text, created_at, author_role
            FROM clinical_notes
                    WHERE encounter_id = ?
            ORDER BY created_at DESC
            """, (encounter_id,))  # Execute a parameterized query to get clinical notes for an encounter
    print(f"getClinicalNotesForEncounter: Retrieved {len(rows)} rows for encounter_id={encounter_id}")  # Log the number of rows retrieved
    return toJson(rows)

#### `getLabsForPatient`

In [62]:
def getLabsForPatient(cursor: sqlite3.Cursor, patient_id: str, test_name: Optional[str] = None, limit: int = 5):
    """"
        Retrieves all lab results associated with a specific patient from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose lab results are being requested.
        Output:
            list: A list of dictionaries, each containing details about a lab result, including:
            - lab_id
            - encounter_id
            - lab_name
            - lab_value
            - lab_units
            - reference_range
            - lab_date
    """

    query = """
            SELECT 
                lab_id, encounter_id, patient_id, lab_name, lab_value, lab_units, reference_range, lab_date
            FROM labs
                    WHERE patient_id = ? 
            """ + (f"AND test_name = ?" if test_name else "") + """
            ORDER BY result_date DESC
            LIMIT ?
            """
    rows = queryDBSafe(cursor, query, (patient_id, limit))  # Execute a parameterized query to get labs for a patient
    print(f"getLabsForPatient: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows)

#### `getMedicationsForPatient`

In [63]:
@tool("getMedicationsForPatient")
def getMedicationsForPatient(cursor: sqlite3.Cursor, patient_id: str, status: str = "active"):
    """
        Retrieves all medication records associated with a specific patient from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose medication records are being requested.
            status (str): Filter medications by status ("active", "stopped", or "all"). Default is "active".
        Output:
            list: A list of dictionaries, each containing details about a medication record, including:
            - medication_id
            - encounter_id
            - medication_name
            - dosage
            - frequency
            - route
            - start_date
            - end_date
    """
    
    if status not in ("active", "stopped", "all"):
        status = "active"

    rows = queryDBSafe(cursor, 
            """
            SELECT 
                 med_id, rxnorm_code, med_name, dose, route, frequency,
                start_date, end_date, status, indication, prescriber_specialty
            FROM medications
                    WHERE patient_id = ?
                    """ + (f"AND status = ?" if status != "all" else "") + """
            ORDER BY start_date DESC
            """, (patient_id, status))  # Execute a parameterized query to get medications for a patient
    print(f"getMedicationsForPatient: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows)

#### `getAllergiesForPatient`

In [64]:
@tool("getAllergiesForPatient")
def getAllergiesForPatient(cursor: sqlite3.Cursor, patient_id: str):
    """"
        Retrieves all allergy records associated with a specific patient from the database.
        Args:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            patient_id (str): The unique identifier of the patient whose allergy records are being requested.
        Output:
            list: A list of dictionaries, each containing details about an allergy record, including:
            - allergy_id
            - allergen
            - reaction
            - severity
            - recorded_date
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
                allergy_id, substance, reaction, severity, recorded_date
            FROM allergies
                    WHERE patient_id = ?
            ORDER BY recorded_date DESC
            """, (patient_id,))  # Execute a parameterized query to get allergies for a patient
    print(f"getAllergiesForPatient: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows)

#### `lookupLabEducation`

In [65]:
@tool("lookupLabEducation")
def lookupLabEducation(test_name: str) -> str:
    """Looks up patient-friendly explanations for lab tests from the reference CSV data.
        Inputs:
            lab_name (str): The name of the lab test to look up (e.g., "Hemoglobin A1c").
        Output:
            Optional[str]: A patient-friendly explanation of the lab test, or None if not found.
    """
    normalized_lab_name = normalizeText(test_name)  # Normalize the input lab name for consistent matching
    df = labExplain.copy()
    df["_k"] = df["test_name_normalized"].astype(str).map(normalizeText)

    hit = df[df["_k"] == normalized_lab_name]
    if hit.empty:
        hit = df[df["_k"].str.contains(normalized_lab_name, na=False)]

    if hit.empty:
        return toJson({"error": f"No lab education found for '{test_name}'"})

    row = hit.iloc[0].drop(labels=["_k"]).to_dict()
    return toJson(row)

#### `lookupMedicationEducation`

In [66]:
@tool("lookupMedicationEducation")
def lookupMedicationEducation(med_name: str) -> str:
    """Looks up patient-friendly explanations for medications from the reference CSV data.
        Inputs:
            med_name (str): The name of the medication to look up (e.g., "Metformin").
        Output:
            Optional[str]: A patient-friendly explanation of the medication, or None if not found.
    """
    normalized_med_name = normalizeText(med_name)  # Normalize the input medication name for consistent matching
    df = medEdu.copy()
    df["_k"] = df["med_name_normalized"].astype(str).map(normalizeText)

    hit = df[df["_k"] == normalized_med_name]
    if hit.empty:
        hit = df[df["_k"].str.contains(normalized_med_name, na=False)]

    if hit.empty:
        return toJson({"error": f"No medication education found for '{med_name}'"})

    row = hit.iloc[0].drop(labels=["_k"]).to_dict()
    return toJson(row)

#### `lookupTrustedSource`

In [67]:
@tool("lookupTrustedSource")
def lookupTrustedSource(source_id: str) -> str:
    """Looks up information about a trusted source from the reference CSV data.
        Inputs:
            source_id (str): Identifier for the trusted source (e.g., "501").
        Output:
            Optional[str]: A description of the trusted source, or None if not found.
    """
    hit = trustedSources[trustedSources["source_id"] == source_id]
    if hit.empty:
        return toJson({"error": f"source_id {source_id} not found"})
    return toJson(hit.iloc[0].to_dict())

#### `policyRoute`

In [68]:
@tool("policyRoute")
def policyRoute(user_text: str) -> str:
    """
    Apply safety guardrails to a user query using `safety_policy_rules.csv` and return a routing decision.

    This tool implements a lightweight, deterministic policy layer that helps the agent decide whether to:
    - answer normally ("answer"),
    - refuse unsafe requests such as diagnosis/dose changes ("refuse"),
    - escalate urgent symptom descriptions ("escalate_emergency" or "escalate_clinician").

    Matching approach (MVP):
    - Normalizes the user text and checks whether each rule's `pattern_or_topic` appears as a substring.
    - If multiple rules match, selects the highest-priority action using:
      escalate_emergency > escalate_clinician > refuse > answer.

    Args:
        user_text (str): The raw user input/query to evaluate.

    Returns:
        str: A JSON string with keys:
            - action: "answer" | "refuse" | "escalate_emergency" | "escalate_clinician"
            - rule_id: matched rule identifier (if any)
            - template: standard response template for refusal/escalation (if any)
            - matched_rules: list of all matched rules with their actions/topics
        If no rules match, returns:
            {"action": "answer", "matched_rules": []}
    """
    normalized_text = normalizeText(user_text)
    rules = policyRules.to_dict(orient="records")

    matches = []
    for r in rules:
        topic = normalizeText(r.get("pattern_or_topic", ""))
        if topic and topic in normalized_text:
            matches.append(r)

    priority = {"escalate_emergency": 3, "escalate_clinician": 2, "refuse": 1, "answer": 0}

    if not matches:
        return toJson({"action": "answer", "matched_rules": []})

    matches_sorted = sorted(
        matches,
        key=lambda r: priority.get(r.get("agent_action", "answer"), 0),
        reverse=True
    )
    best = matches_sorted[0]

    out = {
        "action": best["agent_action"],
        "rule_id": best["rule_id"],
        "template": best["standard_response_template"],
        "matched_rules": [
            {"rule_id": m["rule_id"], "action": m["agent_action"], "topic": m["pattern_or_topic"]}
            for m in matches_sorted
        ]
    }
    return toJson(out)

### Verification for the tools

    Does not work with @tool annotation being present. So the bottom section needs to be commented.

In [69]:
# getPatientInfo(cursor, "P001")
# listPatientEncounters(cursor, "P001", limit=3)
# getRecentClinicalNotes(cursor, "P001", note_type="visit_note")